In [ ]:
from pathlib import Path
import subprocess
from Bio import SeqIO
import shutil

# =====================================================
# settings
# =====================================================

# input genomesا
GENOME_FASTA = Path(r"E:\.....\x.fasta")

# output directory
OUTPUT_DIR = Path(r"E:\.......\ORFs")

# min ORF length based on amino acid
MIN_ORF_AA = 75

# 
GETORF = "getorf"

# =====================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

records = list(SeqIO.parse(GENOME_FASTA, "fasta"))

print(f"\nFound {len(records)} genomes\n")

for i, rec in enumerate(records, start=1):

    genome = rec.id.replace("|","_").replace(":","_").replace("/","_")

    print(f"[{i}/{len(records)}] {genome}")

    # -----------------------------
    # 
    # -----------------------------
    temp_fasta = OUTPUT_DIR / f"{genome}.fasta"

    SeqIO.write(rec, temp_fasta, "fasta")

    # WSL

    wsl_input = "/mnt/e/" + str(temp_fasta.relative_to("E:\\")).replace("\\","/")

    faa_out = OUTPUT_DIR / f"{genome}_orfs.faa"
    fna_out = OUTPUT_DIR / f"{genome}_orfs.fna"

    wsl_faa = "/mnt/e/" + str(faa_out.relative_to("E:\\")).replace("\\","/")
    wsl_fna = "/mnt/e/" + str(fna_out.relative_to("E:\\")).replace("\\","/")

    # =================================================
    # protein output
    # =================================================

    cmd = [
        "wsl",
        GETORF,
        "-sequence", wsl_input,
        "-outseq", wsl_faa,
        "-table", "0",
        "-find", "1",
        "-minsize", str(MIN_ORF_AA*3)
    ]

    subprocess.run(cmd, check=True)

    # =================================================
    # nucleotide output
    # =================================================

    cmd = [
        "wsl",
        GETORF,
        "-sequence", wsl_input,
        "-outseq", wsl_fna,
        "-table", "0",
        "-find", "3",
        "-noreverse",
        "-minsize", str(MIN_ORF_AA*3)
    ]

    subprocess.run(cmd, check=True)

    # delete temprrary files
    temp_fasta.unlink()

print("\nFinished successfully.")